# Phase 20 — interpretable multiscale regime baseline

This notebook defines the deterministic causal comparator that every probabilistic model must beat in usefulness, not merely rename. It separates **structural shape** from **direction** and does not use future outcomes or returns to assign a regime.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    FeatureTier,
    ObservationClock,
    REGIME_ORDER,
    build_multiscale_state,
    dwell_lengths,
    load_close,
    model_frame,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    transition_matrix,
)

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
FIRING_WINDOW_BARS = 240
LAST_N = 12_000

ModuleNotFoundError: No module named 'dc_basics'

## Frozen direct taxonomy

The primary label has four direction-neutral structural classes:

- `broad_coherence`: no adjacent disagreement;
- `shallow_counter`: one boundary and at most half the scales have joined the small-scale counter-move;
- `deep_counter`: one boundary and more than half the scales have joined it;
- `fragmented`: two or more boundaries, so one propagation-depth number is invalid.

`outer_mode`, `small_mode`, and `directed_regime` retain direction separately. The 50% boundary is specified before any outcome analysis and generalizes to arbitrary $N$.

In [ ]:
prices = load_close(DATA_ROOT, SYMBOL)
catalog, global_state, scale_state = build_multiscale_state(
    prices, THRESHOLDS, firing_window=FIRING_WINDOW_BARS, include_scale_state=True
)
print(f"{SYMBOL}: {len(prices):,} prices; {len(global_state):,} initialized hierarchy bars")
print(f"Range: {global_state.index[0]} to {global_state.index[-1]}")
display(catalog.style.format({"threshold": "{:.4%}", "ratio_to_lower": "{:.3f}"}))

## Baseline occupancy under both observation clocks

Bar time measures persistence. Union-event time measures the population at bars where at least one threshold changed mode. Neither is treated as the hidden truth.

In [ ]:
bar_occupancy = global_state["structural_regime"].value_counts(normalize=True).reindex(REGIME_ORDER, fill_value=0.0)
event_state = global_state.loc[global_state["union_event"]]
event_occupancy = event_state["structural_regime"].value_counts(normalize=True).reindex(REGIME_ORDER, fill_value=0.0)
occupancy = pd.concat({
    ObservationClock.BAR.value: bar_occupancy,
    ObservationClock.UNION_EVENT.value: event_occupancy,
}, axis=1)
directed_occupancy = global_state["directed_regime"].value_counts(normalize=True).rename("bar_share")
display(occupancy.style.format("{:.2%}"))
display(directed_occupancy.to_frame().style.format("{:.2%}"))

## What each direct regime contains now

These are contemporaneous causal measurements. The table deliberately contains no forward return, later propagation, rejection, or terminal-depth field.

`counter_depth_fraction` is defined only when the hierarchy has zero or one boundary. A fragmented row has two or more boundaries and therefore no single counter depth, so the field is missing there by design and its `count` is zero for `fragmented`. It is a direct-baseline descriptor, never a model input, so the missing value cannot remove fragmented rows from the later models.

In [ ]:
current_columns = [
    "fragmentation",
    "counter_depth_fraction",
    "mean_mode_age_bars",
    "mean_reversal_progress",
    "max_reversal_progress",
    "hierarchy_dcc_rate",
]
current_profile = global_state.groupby("structural_regime", observed=True)[current_columns].agg(
    ["count", "median", "mean"]
).reindex(REGIME_ORDER)
display(current_profile)

direction_balance = pd.crosstab(
    global_state["structural_regime"], global_state["outer_mode"], normalize="index"
).reindex(REGIME_ORDER)
display(direction_balance.style.format("{:.2%}"))

## Persistence and transitions are descriptions, not model assumptions

In [ ]:
baseline_codes = global_state["structural_regime_code"].to_numpy(dtype=np.int64)
baseline_transition = pd.DataFrame(
    transition_matrix(baseline_codes, len(REGIME_ORDER)), index=REGIME_ORDER, columns=REGIME_ORDER
)
dwell_records = []
for regime_code, regime_name in enumerate(REGIME_ORDER):
    runs = []
    states = baseline_codes
    starts = np.flatnonzero(np.r_[True, states[1:] != states[:-1]])
    ends = np.r_[starts[1:], len(states)]
    for start, end in zip(starts, ends, strict=True):
        if states[start] == regime_code:
            runs.append(end - start)
    dwell_records.append({
        "structural_regime": regime_name,
        "runs": len(runs),
        "median_dwell_bars": np.median(runs) if runs else np.nan,
        "p90_dwell_bars": np.quantile(runs, 0.90) if runs else np.nan,
    })
display(baseline_transition.style.format("{:.3f}"))
display(pd.DataFrame.from_records(dwell_records).set_index("structural_regime"))

## Continuous emission tiers for later models

The direct taxonomy keeps the full discrete hierarchy: ordered modes, boundary count, fragmentation, direction, and event counts. Those fields remain available for transparent rules and diagnostics, but they are not fed to diagonal continuous densities. Otherwise a Gaussian or Student-t component can gain artificial likelihood by pinning a category at its variance floor.

The model tiers therefore contain causal continuous measurements only. On training rows only, three kinds of feature are removed: training-constant features, features with fewer than three distinct values, and zero-inflated features in which one repeated value covers at least 20% of rows. The last case is common at union events: the scale that just fired has mode age 0 and reversal progress 0, and a continuous density would isolate that point mass at its variance floor. Every removal is listed with its reason, and every dropped row is counted by direct structural class. A class present before preparation may not disappear silently.

In [ ]:
tier_records = []
event_source = global_state.loc[global_state["union_event"]]
event_counts = event_source["structural_regime"].value_counts().reindex(REGIME_ORDER, fill_value=0)
for tier in FeatureTier:
    _, proposed_columns = model_frame(global_state, tier)
    prepared_tier = prepare_data(global_state, ObservationClock.UNION_EVENT, tier)
    prepared_counts = prepared_tier.frame["structural_regime"].value_counts().reindex(REGIME_ORDER, fill_value=0)
    lost_regimes = event_counts.index[(event_counts > 0) & prepared_counts.eq(0)].tolist()
    if lost_regimes:
        raise AssertionError(f"Feature preparation removed structural classes: {lost_regimes}")
    tier_records.append({
        "tier": tier.value,
        "proposed_features": len(proposed_columns),
        "retained_features": len(prepared_tier.feature_columns),
        "removed_features": prepared_tier.removed_feature_reasons,
        "rows_dropped": prepared_tier.dropped_rows,
        "dropped_regime_counts": prepared_tier.dropped_regime_counts,
        "columns": prepared_tier.feature_columns,
    })
display(pd.DataFrame.from_records(tier_records).set_index("tier"))

for forbidden in (
    "future", "terminal", "outcome", "resolution", "mode_sign", "fragmentation",
    "counter_depth", "direction_boundaries", "dcc_rate",
):
    if any(forbidden in column for record in tier_records for column in record["columns"]):
        raise AssertionError(f"Forbidden continuous-emission feature token found: {forbidden}")
print("Causality, class-retention, and continuous-emission audits: passed")

In [ ]:
plot_price_states(
    global_state,
    "directed_regime",
    f"{SYMBOL}: direct causal multiscale regime baseline",
    last_n=LAST_N,
).show()

# Phase 20 completion gate

Before probabilistic modelling, be able to explain:

1. Why structural shape and direction are separate axes.
2. Why broad coherence is not automatically a profitable trend.
3. Why shallow and deep counter-moves are defined by a scale fraction rather than future propagation.
4. Why fragmented structure has no valid single propagation depth, and why fragmented rows must not be lost in preprocessing.
5. Why bar-time and event-time occupancy differ.
6. Why the direct taxonomy is a comparator, not ground truth.
7. Why discrete hierarchy geometry stays in the direct comparator instead of entering continuous emission densities.
8. Why later models must add probabilistic confidence, persistence, compression, or useful separation rather than merely reproduce these names.